In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: C:\Users\Soumadipta Konar\Desktop\fitrank-ai


In [2]:
import pandas as pd

from rank import run_pipeline

C:\Users\Soumadipta Konar\anaconda3\envs\torchgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
RAW_DIR = ROOT / "data" / "raw"
OUT_PATH = ROOT / "data" / "output" / "submission.csv"
DEBUG_PATH = ROOT / "data" / "output" / "top100_debug.csv"

print("Raw data:", RAW_DIR)
print("Submission:", OUT_PATH)
print("Debug:", DEBUG_PATH)

Raw data: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\raw
Submission: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\output\submission.csv
Debug: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\output\top100_debug.csv


In [4]:
import inspect
from src.semantic_ranker import compute_semantic_scores

print(inspect.signature(compute_semantic_scores))

(df, jd_text, model_name='sentence-transformers/all-MiniLM-L6-v2', device='auto', batch_size=256, top_k=25000, candidate_indices=None)


In [5]:
submission = run_pipeline(
    raw_dir=RAW_DIR,
    out_path=OUT_PATH,
    debug_path=DEBUG_PATH,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    device="cuda",
    batch_size=256,
    top_k=25000,
    candidate_pool=30000,
)

FitRank AI: Optimized Hybrid Semantic Candidate Ranking

[1/9] Finding competition files...
Candidates: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\raw\[PUB] India_runs_data_and_ai_challenge\India_runs_data_and_ai_challenge\candidates.jsonl
Job description: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\raw\[PUB] India_runs_data_and_ai_challenge\India_runs_data_and_ai_challenge\job_description.docx
Validator: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\raw\[PUB] India_runs_data_and_ai_challenge\India_runs_data_and_ai_challenge\validate_submission.py

[2/9] Reading job description...
JD characters: 9564

[3/9] Loading candidate profiles...


Loading candidates: 100000it [00:21, 4678.60it/s]


Loaded candidates: 100000

[4/9] Computing TF-IDF lexical scores on all candidates...

[5/9] Computing structured feature scores on all candidates...

[6/9] Selecting candidate pool for semantic ranking...
Selected candidate pool: 30000

[7/9] Computing semantic scores using SentenceTransformer + FAISS on pool...
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding device: cuda
Semantic pool size: 30000 / 100000


Batches: 100%|██████████| 118/118 [01:03<00:00,  1.85it/s]



[8/9] Computing final ranking...

Top 10 candidates:
candidate_id                            title  years_experience               location  final_score_raw  semantic_score  lexical_score  ir_nlp_score  skill_depth_score  behavior_score  honeypot_penalty
CAND_0018499 Senior Machine Learning Engineer               7.2   Noida, Uttar Pradesh         0.895999        0.484170       0.963619           1.0           0.975805          0.8334               1.0
CAND_0002025               Senior AI Engineer               5.9     Trivandrum, Kerala         0.895160        0.574877       0.843137           1.0           0.987760          0.8240               1.0
CAND_0046525 Senior Machine Learning Engineer               6.1      Pune, Maharashtra         0.887517        0.560537       0.807180           1.0           0.934577          0.8270               1.0
CAND_0011687              Senior NLP Engineer               7.8 Indore, Madhya Pradesh         0.885896        0.540928       0.760752    

In [6]:
import pandas as pd

submission = pd.read_csv(OUT_PATH)

print(submission.shape)
submission.head(20)

(100, 4)


,candidate_id,rank,score,reasoning
0,CAND_0018499,1,0.995000,Senior Machine Learning Engineer with 7.2 year...
1,CAND_0002025,2,0.992030,Senior AI Engineer with 5.9 years experience; ...
2,CAND_0046525,3,0.964977,Senior Machine Learning Engineer with 6.1 year...
3,CAND_0011687,4,0.959236,Senior NLP Engineer with 7.8 years experience;...
4,CAND_0055905,5,0.952387,Senior Machine Learning Engineer with 8.1 year...
5,CAND_0081846,6,0.922949,Lead AI Engineer with 6.7 years experience; re...
6,CAND_0077337,7,0.902910,Staff Machine Learning Engineer with 7.0 years...
7,CAND_0008425,8,0.884958,Senior NLP Engineer with 7.8 years experience;...
8,CAND_0086022,9,0.864749,Senior Applied Scientist with 5.3 years experi...
9,CAND_0053591,10,0.861245,AI Engineer with 5.3 years experience; relevan...


In [7]:
debug = pd.read_csv(DEBUG_PATH)

debug[[
    "rank",
    "candidate_id",
    "title",
    "years_experience",
    "location",
    "score",
    "semantic_score",
    "lexical_score",
    "ir_nlp_score",
    "skill_depth_score",
    "production_score",
    "behavior_score",
    "honeypot_penalty",
    "reasoning"
]].head(20)

,rank,candidate_id,title,years_experience,location,score,semantic_score,lexical_score,ir_nlp_score,skill_depth_score,production_score,behavior_score,honeypot_penalty,reasoning
0,1,CAND_0018499,Senior Machine Learning Engineer,7.2,"Noida, Uttar Pradesh",0.995000,0.484170,0.963619,1.0,0.975805,1.0,0.8334,1.0,Senior Machine Learning Engineer with 7.2 year...
1,2,CAND_0002025,Senior AI Engineer,5.9,"Trivandrum, Kerala",0.992030,0.574877,0.843137,1.0,0.987760,1.0,0.8240,1.0,Senior AI Engineer with 5.9 years experience; ...
2,3,CAND_0046525,Senior Machine Learning Engineer,6.1,"Pune, Maharashtra",0.964977,0.560537,0.807180,1.0,0.934577,1.0,0.8270,1.0,Senior Machine Learning Engineer with 6.1 year...
3,4,CAND_0011687,Senior NLP Engineer,7.8,"Indore, Madhya Pradesh",0.959236,0.540929,0.760752,1.0,1.001502,1.0,0.8411,1.0,Senior NLP Engineer with 7.8 years experience;...
4,5,CAND_0055905,Senior Machine Learning Engineer,8.1,London,0.952387,0.571930,1.000000,1.0,0.969051,1.0,0.7023,1.0,Senior Machine Learning Engineer with 8.1 year...
5,6,CAND_0081846,Lead AI Engineer,6.7,"Jaipur, Rajasthan",0.922949,0.567865,0.994876,1.0,0.962231,1.0,0.6401,1.0,Lead AI Engineer with 6.7 years experience; re...
6,7,CAND_0077337,Staff Machine Learning Engineer,7.0,"Kochi, Kerala",0.902910,0.573056,0.577777,1.0,0.971601,1.0,0.8718,1.0,Staff Machine Learning Engineer with 7.0 years...
7,8,CAND_0008425,Senior NLP Engineer,7.8,"Kolkata, West Bengal",0.884958,0.571889,0.744883,1.0,0.977623,1.0,0.7516,1.0,Senior NLP Engineer with 7.8 years experience;...
8,9,CAND_0086022,Senior Applied Scientist,5.3,"Kolkata, West Bengal",0.864749,0.551186,0.732494,1.0,0.955108,1.0,0.7487,1.0,Senior Applied Scientist with 5.3 years experi...
9,10,CAND_0053591,AI Engineer,5.3,Toronto,0.861245,0.596245,0.580892,1.0,0.950358,1.0,0.8673,1.0,AI Engineer with 5.3 years experience; relevan...


In [8]:
print("Rows:", len(submission))
print("Unique candidates:", submission["candidate_id"].nunique())
print("Ranks valid:", submission["rank"].tolist() == list(range(1, 101)))
print("Scores non-increasing:", submission["score"].is_monotonic_decreasing)
print("Missing values:")
print(submission.isna().sum())

Rows: 100
Unique candidates: 100
Ranks valid: True
Scores non-increasing: True
Missing values:
candidate_id    0
rank            0
score           0
reasoning       0
dtype: int64
